In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/mianaftab/rim-of-cars-dataset/rim-detection.ndjson


In [3]:
import json

ndjson_path = "/kaggle/input/datasets/mianaftab/rim-of-cars-dataset/rim-detection.ndjson"

data = []

with open(ndjson_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print("Total records:", len(data))
print("\nFirst record:")
print(json.dumps(data[0], indent=2)[:5000])

Total records: 431

First record:
{
  "type": "dataset",
  "task": "pose",
  "name": "Rim detection",
  "description": "Street-level and studio views of modern passenger cars and SUVs captured outdoors and at motor shows, with annotated wheel poses for accurate orientation analysis.",
  "bytes": 135983188,
  "url": "https://platform.ultralytics.com/mian-aftab/datasets/rim-detection",
  "class_names": {
    "0": "wheel",
    "1": "just rim"
  },
  "version": "latest",
  "created_at": "2026-09-07T08:13:21.422Z",
  "updated_at": "2026-09-07T08:13:21.422Z",
  "kpt_shape": [
    4,
    3
  ]
}


In [4]:
print("Keys in first record:")
print(data[0].keys())

Keys in first record:
dict_keys(['type', 'task', 'name', 'description', 'bytes', 'url', 'class_names', 'version', 'created_at', 'updated_at', 'kpt_shape'])


In [5]:
for key, value in data[0].items():
    print("\nKEY:", key)
    print("TYPE:", type(value))
    
    if isinstance(value, dict):
        print("DICT KEYS:", value.keys())
    elif isinstance(value, list):
        print("LIST LENGTH:", len(value))
        if len(value) > 0:
            print("FIRST ITEM:", value[0])
    else:
        print("VALUE:", value)


KEY: type
TYPE: <class 'str'>
VALUE: dataset

KEY: task
TYPE: <class 'str'>
VALUE: pose

KEY: name
TYPE: <class 'str'>
VALUE: Rim detection

KEY: description
TYPE: <class 'str'>
VALUE: Street-level and studio views of modern passenger cars and SUVs captured outdoors and at motor shows, with annotated wheel poses for accurate orientation analysis.

KEY: bytes
TYPE: <class 'int'>
VALUE: 135983188

KEY: url
TYPE: <class 'str'>
VALUE: https://platform.ultralytics.com/mian-aftab/datasets/rim-detection

KEY: class_names
TYPE: <class 'dict'>
DICT KEYS: dict_keys(['0', '1'])

KEY: version
TYPE: <class 'str'>
VALUE: latest

KEY: created_at
TYPE: <class 'str'>
VALUE: 2026-09-07T08:13:21.422Z

KEY: updated_at
TYPE: <class 'str'>
VALUE: 2026-09-07T08:13:21.422Z

KEY: kpt_shape
TYPE: <class 'list'>
LIST LENGTH: 2
FIRST ITEM: 4


In [6]:
import json

ndjson_path = "/kaggle/input/datasets/mianaftab/rim-of-cars-dataset/rim-detection.ndjson"

records = []

with open(ndjson_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Total records:", len(records))

# Print all record types
from collections import Counter

types = Counter(record.get("type") for record in records)

print("\nRecord types:")
print(types)

Total records: 431

Record types:
Counter({'image': 430, 'dataset': 1})


In [7]:
for i, record in enumerate(records):
    if record.get("type") != "dataset":
        print("Record index:", i)
        print(json.dumps(record, indent=2)[:10000])
        break

Record index: 1
{
  "type": "image",
  "file": "74eb0a162c824632bdb6ecdc84b54984.jpg",
  "url": "https://cdn.ul.run/us/i/00ad3ad805ca35c1150dc194b83d98e5.jpg?Expires=1789708255&KeyName=key-v1&Signature=cY8sICGigtmMFvpgYFiOJrCXdv8",
  "width": 720,
  "height": 405,
  "split": "train",
  "annotations": {
    "pose": [
      [
        0,
        0.48148,
        0.65095,
        0.15344,
        0.33422,
        0.48679,
        0.50382,
        2,
        0.47581,
        0.79808,
        2,
        0.41602,
        0.66281,
        2,
        0.54699,
        0.63941,
        2
      ],
      [
        0,
        0.8379,
        0.57482,
        0.06793,
        0.21305,
        0.8407,
        0.4883,
        2,
        0.83621,
        0.66134,
        2,
        0.8152,
        0.59089,
        2,
        0.86061,
        0.56809,
        2
      ]
    ]
  }
}


In [8]:
import os
import json
import shutil
from collections import Counter

# ============================================================
# 1. PATHS
# ============================================================

NDJSON_PATH = "/kaggle/input/datasets/mianaftab/rim-of-cars-dataset/rim-detection.ndjson"

OUTPUT_DIR = "/kaggle/working/rim_yolo"

# ============================================================
# 2. CREATE YOLO FOLDER STRUCTURE
# ============================================================

for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

print("YOLO folders created.")

# ============================================================
# 3. READ NDJSON
# ============================================================

records = []

with open(NDJSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if line:
            records.append(json.loads(line))

print("Total records:", len(records))

# ============================================================
# 4. GET DATASET INFORMATION
# ============================================================

dataset_info = records[0]

print("Dataset:", dataset_info.get("name"))
print("Task:", dataset_info.get("task"))
print("Classes:", dataset_info.get("class_names"))
print("Keypoint shape:", dataset_info.get("kpt_shape"))

YOLO folders created.
Total records: 431
Dataset: Rim detection
Task: pose
Classes: {'0': 'wheel', '1': 'just rim'}
Keypoint shape: [4, 3]


In [9]:
import requests
from tqdm.auto import tqdm

image_records = [
    r for r in records
    if r.get("type") == "image"
]

print("Number of images:", len(image_records))

Number of images: 430


In [10]:
print(image_records[0]["file"])
print(image_records[0]["url"])
print(image_records[0]["width"], image_records[0]["height"])
print(image_records[0]["split"])

74eb0a162c824632bdb6ecdc84b54984.jpg
https://cdn.ul.run/us/i/00ad3ad805ca35c1150dc194b83d98e5.jpg?Expires=1789708255&KeyName=key-v1&Signature=cY8sICGigtmMFvpgYFiOJrCXdv8
720 405
train


In [11]:
import requests
import os
from tqdm.auto import tqdm

downloaded = 0
failed = 0
label_count = 0

for record in tqdm(image_records, desc="Converting dataset"):

    filename = record["file"]
    url = record["url"]
    split = record.get("split", "train")

    # Make sure split is valid
    if split not in ["train", "val", "test"]:
        split = "train"

    image_path = os.path.join(
        OUTPUT_DIR,
        "images",
        split,
        filename
    )

    label_path = os.path.join(
        OUTPUT_DIR,
        "labels",
        split,
        os.path.splitext(filename)[0] + ".txt"
    )

    # --------------------------------------------------------
    # Download image
    # --------------------------------------------------------

    try:

        response = requests.get(url, timeout=30)

        if response.status_code == 200:

            with open(image_path, "wb") as img_file:
                img_file.write(response.content)

            downloaded += 1

        else:
            print(
                f"Failed image: {filename} "
                f"Status: {response.status_code}"
            )
            failed += 1
            continue

    except Exception as e:

        print(f"Error downloading {filename}: {e}")
        failed += 1
        continue

    # --------------------------------------------------------
    # Create YOLO Pose label
    # --------------------------------------------------------

    pose_annotations = record.get("annotations", {}).get("pose", [])

    with open(label_path, "w") as label_file:

        for annotation in pose_annotations:

            # annotation already contains:
            # class + bbox + 4 keypoints

            values = [str(v) for v in annotation]

            label_file.write(" ".join(values) + "\n")

            label_count += 1


print("\n================================")
print("Conversion completed")
print("================================")
print("Images downloaded:", downloaded)
print("Failed downloads:", failed)
print("Pose objects:", label_count)

Converting dataset:   0%|          | 0/430 [00:00<?, ?it/s]


Conversion completed
Images downloaded: 430
Failed downloads: 0
Pose objects: 642


In [12]:
from collections import Counter
import os

for split in ["train", "val", "test"]:

    image_dir = f"{OUTPUT_DIR}/images/{split}"
    label_dir = f"{OUTPUT_DIR}/labels/{split}"

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.endswith(".txt")
    ]

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))


TRAIN
Images: 179
Labels: 342

VAL
Images: 45
Labels: 86

TEST
Images: 0
Labels: 0


In [13]:
import glob

label_files = glob.glob(
    f"{OUTPUT_DIR}/labels/train/*.txt"
)

print("Number of train labels:", len(label_files))

if label_files:

    print("\nExample label:")
    
    with open(label_files[0], "r") as f:
        print(f.read())

Number of train labels: 342

Example label:
0 0.5192 0.75816 0.0846 0.2244 0.52374 0.66596 2 0.51646 0.85035 2 0.48817 0.768 2 0.55029 0.74461 2
0 0.7783 0.63799 0.05983 0.18247 0.78077 0.56677 2 0.77523 0.70927 2 0.75967 0.64752 2 0.79698 0.62573 2



In [14]:
yaml_content = f"""
path: {OUTPUT_DIR}

train: images/train
val: images/val
test: images/test

kpt_shape: [4, 3]

names:
  0: wheel
  1: just rim
"""

yaml_path = f"{OUTPUT_DIR}/data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_content)
print("Saved:", yaml_path)


path: /kaggle/working/rim_yolo

train: images/train
val: images/val
test: images/test

kpt_shape: [4, 3]

names:
  0: wheel
  1: just rim

Saved: /kaggle/working/rim_yolo/data.yaml


In [15]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.1 MB/s eta 0:00:00


In [16]:
from ultralytics import YOLO

print("Ultralytics imported successfully")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics imported successfully


In [17]:
from ultralytics import YOLO

model = YOLO("yolo26n-pose.pt")

results = model.train(
    data="/kaggle/working/rim_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=[0,1],
    name="rim_pose"
)

Ultralytics 8.4.147 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rim_yolo/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt,

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss    l1_loss   rle_loss  Instances       Size
      2/100      1.62G      1.138      3.392     0.6913      4.758    0.01594      16.17         13        640: 100% ━━━━━━━━━━━━ 22/22 2.7it/s 8.1s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 5.0it/s 0.6s0.3s
                   all         86        131      0.569      0.895       0.63      0.314      0.289      0.429      0.267     0.0619

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss    l1_loss   rle_loss  Instances       Size
      3/100      1.62G      1.224      2.627      0.679      4.118    0.01834      11.77          9        640: 100% ━━━━━━━━━━━━ 22/22 2.6it/s 8.4s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 4.5it/s 